# 物体検出

**物体検出（object detection）** は、画像中に写っている物体について「どこに」「何が」写っているかを推定するタスク。各物体を囲む矩形領域（バウンディングボックス）の位置と、その物体のクラスラベルの両方を出力する。

| タスク | 出力 |
| --- | --- |
| 画像分類（classification） | 画像全体に対する1つのクラスラベル |
| 物体検出（object detection） | 物体ごとのバウンディングボックス＋クラスラベル |
| セマンティックセグメンテーション（semantic segmentation） | ピクセルごとのクラスラベル（個体は区別しない） |
| インスタンスセグメンテーション（instance segmentation） | 物体ごとのピクセル単位マスク＋クラスラベル |

画像分類は「画像→記述」の写像だが、物体検出は1枚の画像から可変個の物体を検出する必要があるため、単純な分類問題より難しい。この可変長出力をどう扱うかという観点が、後続の各アルゴリズム（R-CNN系、YOLO/SSD系、FCOS、DETR系）の設計思想の違いにつながる。

---

**目次**

:::{toc}
:context: children
:depth: 1
:::

---

## タスクの定義

物体検出モデルは、画像 $I$ を入力として、物体の集合

$$
\{(b_i, c_i, s_i)\}_{i=1}^{N}
$$

を出力する。ここで

- $b_i = (x_{\min}, y_{\min}, x_{\max}, y_{\max})$：$i$番目のバウンディングボックスの座標
- $c_i$：クラスラベル
- $s_i$：信頼度スコア（confidence score）
- $N$：検出された物体数（画像ごとに可変）

というように、「可変個の (位置, クラス, スコア) の組」を予測する点が、固定長のベクトルを出力する分類・回帰と大きく異なる。

学習時の正解データ（ground truth）も同様に、画像ごとに $\{(b_i^{gt}, c_i^{gt})\}$ という可変個のボックス集合として与えられる。予測と正解をどう対応づけるか（マッチング）、対応づけられなかった予測やボックスをどう扱うかが、モデル設計上の主要な論点になる。

## 検出パラダイムの概観

評価指標が共通していても、「どのように候補領域を生成し、どのように分類・回帰するか」という設計は手法によって大きく異なる。本ノートブック以下では、次の系統に分けて整理する。

| 系統 | 代表手法 | 基本アイデア |
| --- | --- | --- |
| Two-stage（二段階） | R-CNN, Fast R-CNN, Faster R-CNN | 領域候補を提案してから分類・回帰する |
| One-stage（一段階） | YOLO, SSD | グリッド／アンカーに対して1回の forward で直接予測する |
| Anchor-free | FCOS など | アンカーを使わずピクセル単位で直接予測する |
| Transformer系 | DETR, Deformable DETR, RT-DETR | 集合予測問題として定式化し、NMSなどの後処理を排除する |